In [1]:
pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 6.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import category_encoders as ce

In [3]:
df=pd.read_csv(r'')
df

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,City,Gender,Country,Clicked on Ad,Unnamed: 8
0,62.26,32,69481.85,172.83,Lisafort,Male,Svalbard & Jan Mayen Islands,0,NaN
1,41.73,31,61840.26,207.17,West Angelabury,Male,Singapore,0,NaN
2,44.40,30,57877.15,172.83,Reyesfurt,Female,Guadeloupe,0,NaN
3,59.88,28,56180.93,207.17,New Michael,Female,Zambia,0,NaN
4,49.21,30,54324.73,201.58,West Richard,Female,Qatar,1,NaN
...,...,...,...,...,...,...,...,...,...
9995,41.73,31,61840.26,207.17,West Angelabury,Male,Singapore,1,NaN
9996,41.73,28,51501.38,120.49,Kennedyfurt,Male,Luxembourg,0,NaN
9997,55.60,39,38067.08,124.44,North Randy,Female,Egypt,0,NaN
9998,46.61,50,43974.49,123.13,North Samantha,Female,Malawi,1,NaN


In [4]:
#REMOVING THE EXTRA COLUMN

df=df.drop("Unnamed: 8", axis=1)


In [5]:
#REMOVING DUPLICATES

df=df.drop_duplicates()

In [6]:
#CHECK FOR MISSING VALUES

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9444 entries, 0 to 9997
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Daily Time Spent on Site  9444 non-null   float64
 1   Age                       9444 non-null   int64  
 2   Area Income               9444 non-null   float64
 3   Daily Internet Usage      9444 non-null   float64
 4   City                      9444 non-null   object 
 5   Gender                    9444 non-null   object 
 6   Country                   9444 non-null   object 
 7   Clicked on Ad             9444 non-null   int64  
dtypes: float64(3), int64(2), object(3)
memory usage: 664.0+ KB


In [7]:
#ENCODING CATEGORICAL COLUMNS

encoder = ce.TargetEncoder(cols=['Country'])
df['Country'] = encoder.fit_transform(df['Country'], df['Clicked on Ad'])

encoder = ce.TargetEncoder(cols=['City'])
df['City'] = encoder.fit_transform(df['City'], df['Clicked on Ad'])

encoder = ce.TargetEncoder(cols=['Gender'])
df['Gender'] = encoder.fit_transform(df['Gender'], df['Clicked on Ad'])

In [8]:
df

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,City,Gender,Country,Clicked on Ad
0,62.26,32,69481.85,172.83,0.030151,0.457654,0.392258,0
1,41.73,31,61840.26,207.17,0.136010,0.457654,0.008427,0
2,44.40,30,57877.15,172.83,0.258883,0.515825,0.366868,0
3,59.88,28,56180.93,207.17,0.009723,0.515825,0.130536,0
4,49.21,30,54324.73,201.58,0.593735,0.515825,0.037383,1
...,...,...,...,...,...,...,...,...
9993,69.62,34,59785.94,231.94,0.041890,0.457654,0.909688,0
9994,41.73,28,39799.73,217.37,0.176164,0.515825,0.130536,0
9995,41.73,31,61840.26,207.17,0.136010,0.457654,0.008427,1
9996,41.73,28,51501.38,120.49,0.250097,0.457654,0.335665,0


In [9]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

data=df

X = data.drop('Clicked on Ad', axis=1)
y = data['Clicked on Ad']

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
columns = ['Daily Time Spent on Site','Age','Area Income','Daily Internet Usage']
X[columns] = scaler.fit_transform(X[columns])

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create instance of XGBoost model
model_xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.1, 0.2, 0.3],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

# Perform Grid Search with cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(estimator=model_xgb, param_grid=param_grid, cv=kf, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best model
best_model = grid_search.best_estimator_

# Cross-validation scores with the best model
scores = cross_val_score(best_model, X, y, cv=kf, scoring='accuracy')

# Model Training with the best model
best_model.fit(X_train, y_train)

# Make predictions using the best model
y_pred = best_model.predict(X_test)

# Performance metrics
average_accuracy = np.mean(scores)
print("XGBOOST MODEL")
print(f"Accuracy Score for each fold: {[round(score, 4) for score in scores]}")


accuracy2 = accuracy_score(y_test, y_pred)
print(f"Accuracy on test data: {accuracy2:.2f}")

report = classification_report(y_test, y_pred)
print(report)
print(f"Best Parameters: {grid_search.best_params_}")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:15:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:15:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:15:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:15:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

XGBOOST MODEL
Accuracy Score for each fold: [np.float64(0.8883), np.float64(0.8761), np.float64(0.8804), np.float64(0.8788), np.float64(0.8824)]
Accuracy on test data: 0.88
              precision    recall  f1-score   support

           0       0.89      0.88      0.89       980
           1       0.88      0.88      0.88       909

    accuracy                           0.88      1889
   macro avg       0.88      0.88      0.88      1889
weighted avg       0.88      0.88      0.88      1889

Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.9}


In [11]:
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier()

k=5
kf= KFold(n_splits=k, shuffle=True, random_state=42)
scores = cross_val_score(model_rf, X, y, cv=kf, scoring='accuracy')

# Train the model
model_rf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model_rf.predict(X_test)


# performance metrics
average_accuracy = np.mean(scores)
print(f"Accuracy Score for each fold: {[round(score, 4) for score in scores]}")
print(f"Average accuracy across {k} folds: {average_accuracy:.2f}")

accuracy1 = accuracy_score(y_test, y_pred)
print(f"RANDOM FOREST-Accuracy: {accuracy1:.2f}")
report = classification_report(y_test, y_pred)
print(report)

Accuracy Score for each fold: [np.float64(0.8767), np.float64(0.8671), np.float64(0.8735), np.float64(0.8698), np.float64(0.8824)]
Average accuracy across 5 folds: 0.87
RANDOM FOREST-Accuracy: 0.88
              precision    recall  f1-score   support

           0       0.89      0.87      0.88       980
           1       0.87      0.88      0.87       909

    accuracy                           0.88      1889
   macro avg       0.88      0.88      0.88      1889
weighted avg       0.88      0.88      0.88      1889



In [12]:
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier()

# Train the model
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"DECISION TREE-Accuracy: {accuracy:.2f}")
report = classification_report(y_test, y_pred)
print(report)

DECISION TREE-Accuracy: 0.83
              precision    recall  f1-score   support

           0       0.84      0.83      0.83       980
           1       0.82      0.82      0.82       909

    accuracy                           0.83      1889
   macro avg       0.83      0.83      0.83      1889
weighted avg       0.83      0.83      0.83      1889



In [13]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()

# Train the model
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"LOGISTIC REGRESSION-Accuracy: {accuracy:.2f}")
report = classification_report(y_test, y_pred)
print(report)

LOGISTIC REGRESSION-Accuracy: 0.87
              precision    recall  f1-score   support

           0       0.87      0.88      0.88       980
           1       0.87      0.86      0.87       909

    accuracy                           0.87      1889
   macro avg       0.87      0.87      0.87      1889
weighted avg       0.87      0.87      0.87      1889

